In [84]:
from absl import app, logging
import pdb
import torch, torchaudio, argparse, os, tqdm, re, gin
import cached_conv as cc
from rave.core import get_rave_receptive_field
from rave.masker import SpectrogramMasking
try:
    import rave
except:
    import sys, os 
    sys.path.append(os.path.abspath('.'))
    import rave

def get_audio_files(path):
    audio_files = []
    valid_exts = rave.core.get_valid_extensions()
    for root, _, files in os.walk(path):
        valid_files = list(filter(lambda x: os.path.splitext(x)[1] in valid_exts, files))
        audio_files.extend([(path, os.path.join(root, f)) for f in valid_files])
    return audio_files


@torch.no_grad()
def main():
    torch.set_float32_matmul_precision('high')

    cc.use_cached_conv(False)

    model_path = './pretrained/ctest/ctest_c1_r10-drumset'

    # load model
    logging.info("building rave")
    is_scripted = False
    if not os.path.exists(model_path):
        logging.error('path %s does not seem to exist.'%model_path)
        exit()
    if os.path.splitext(model_path)[1] == ".ts":
        model = torch.jit.load(model_path)
        is_scripted = True
    else:
        config_path = rave.core.search_for_config(model_path)
        print(f'Gin config path is {config_path}')
        if config_path is None:
            logging.error('config not found in folder %s'%model_path)
        gin.parse_config_file(config_path)
        model = rave.RAVE()
        run = rave.core.search_for_run(model_path)
        if run is None:
            logging.error("run not found in folder %s"%model_path)
        model = model.load_from_checkpoint(run)
        model = model.eval()
        pqmf_channels = model.pqmf.forward_conv.weight.shape[0]

    device = torch.device('cpu')

    # parse inputs
    ratio = rave.core.get_minimum_size(model)
    print(f'[INFO] Compression ratio is {ratio} samples')

    masker = SpectrogramMasking(target_type='relative_power',
                                pow_times=1,
                                win_length=256,
                                hop_ratio=256//4,
                                mask_ratio=0)
    # clean cache
    _ = model(torch.zeros(1,1,2**16))
    return model, masker



In [85]:

model,masker = main()

Gin config path is /Users/franco/aim/projs/RAVE/pretrained/ctest/ctest_c1_r10-drumset/config.gin
!!!!!!Forward padding (512, 0)
!!!!!!Forward padding (512, 0)
[INFO] Compression ratio is 256 samples


In [88]:
audio, sr = torchaudio.load('./experiments/test_audios/drumset/7_pop-groove7_138_beat_4-4_1.wav')
if sr != model.sr:
    audio = torchaudio.functional.resample(audio, sr, model.sr)

audio = audio[:,44100*0:44100*9]
audio = audio[:,audio.shape[1]%2048:]

In [93]:
with torch.no_grad():
    masker.mask_ratio = 0.00
    x = masker(audio)
    print(f'Masker {x.shape}')
    out = model.forward(x[None])
    print(f'Output {out.shape}')
    distance = model.audio_distance(out,audio.unsqueeze(1))['spectral_distance']
    print('Distance ',distance)
    ipd.display(ipd.Audio(x, rate=sr))
    ipd.display(ipd.Audio(out.squeeze(1), rate=sr))

Masker torch.Size([1, 395264])
Output torch.Size([1, 1, 395264])
Distance  tensor(13.1925)


In [83]:
with torch.no_grad():
    masker.mask_ratio = 0.1
    x = masker(audio)
    print(f'Masker {x.shape}')
    out = model.forward(x[None])
    print(f'Output {out.shape}')
    distance = model.audio_distance(out,audio.unsqueeze(1))['spectral_distance']
    print('Distance ',distance)
    ipd.display(ipd.Audio(x, rate=sr))
    ipd.display(ipd.Audio(out.squeeze(1), rate=sr))

Masker torch.Size([1, 176128])
Output torch.Size([1, 1, 176128])
Distance  tensor(34.9361)


In [40]:
with torch.no_grad():
    masker.mask_ratio = 1
    x = masker(audio)
    print(f'Masker {x.shape}')
    out = model.forward(x[None])
    print(f'Output {out.shape}')
    distance = model.audio_distance(out,audio.unsqueeze(1))['spectral_distance']
    print('Distance ',distance)
    ipd.display(ipd.Audio(x, rate=sr))
    ipd.display(ipd.Audio(out.squeeze(1), rate=sr))

Masker torch.Size([1, 176128])
Output torch.Size([1, 1, 176128])
Distance  tensor(988.4836)


/Users/franco/tools/miniconda3/envs/rave/lib/python3.9/site-packages/IPython/lib/display.py:187: RuntimeWarning: invalid value encountered in divide
  scaled = data / normalization_factor * 32767
/Users/franco/tools/miniconda3/envs/rave/lib/python3.9/site-packages/IPython/lib/display.py:188: RuntimeWarning: invalid value encountered in cast
  return scaled.astype("<h").tobytes(), nchan
